# Setup

All the Imports needed for this experiment.

In [ ]:
from matplotlib import pyplot as plt
from collections import defaultdict
import gymnasium as gym
import numpy as np
import ipywidgets as widgets
from datetime import datetime

import sys
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "env" / "blackjack_env.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from env.blackjack_env import BlackjackEnv
from agents.q_learning_agent import QLearningBlackjackAgent


Define default font size, plot colors etc.

In [ ]:
def setup_plot_style():
    """Definiert das globale Design für alle Matplotlib-Plots."""
    # Schriftgrößen
    plt.rcParams['font.size'] = 11
    plt.rcParams['axes.titlesize'] = 14
    plt.rcParams['axes.titleweight'] = 'bold'
    plt.rcParams['axes.labelsize'] = 12
    plt.rcParams['legend.fontsize'] = 11
    plt.rcParams['xtick.labelsize'] = 10
    plt.rcParams['ytick.labelsize'] = 10
    
    # Linien & Grid
    plt.rcParams['lines.linewidth'] = 2.0
    plt.rcParams['axes.grid'] = True
    plt.rcParams['grid.linestyle'] = '--'
    plt.rcParams['grid.alpha'] = 0.5
    
    # Layout & Render-Qualität
    plt.rcParams['figure.autolayout'] = True
    plt.rcParams['figure.dpi'] = 120
    
    # Optionale Farbpalette (z.B. "Tab10" oder "Set2" für modernen Look)
    plt.style.use('seaborn-v0_8-whitegrid') # Falls ein Grundtheme gewünscht ist

setup_plot_style()
AGENT_STYLES = {    
    "baseline": {"color": "#6b7c93", "label": "Baseline"},
    "counting":  {"color": "#2a9d8f", "label": "Counting"},
}


Create image folder for high-res images and image save function.

In [ ]:
IMAGES_PATH = PROJECT_ROOT / "images"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)
    print(f"Figure saved: {path}")

# Environment & Agents

In [ ]:
def baseline_state_key(obs) -> tuple[int, int, int]:
    player_total, dealer_upcard, usable_ace = obs[:3]

    return (
        int(player_total),
        int(dealer_upcard),
        int(usable_ace)
    )

def counting_state_key(obs: np.ndarray):
    import numpy as np
    player_total, dealer_upcard, usable_ace, running_count, true_count, cards_remaining = obs

    return (
        int(player_total),
        int(dealer_upcard),
        int(usable_ace),

        int(np.clip(running_count, -20, 20)),
        int(np.clip(true_count, -10, 10)),
        int(np.clip(cards_remaining // 52, 0, 6)),
    )


class BaselineBlackjackAgent(QLearningBlackjackAgent):
    def __init__(
            self, 
            env: gym.Env,
            **kwargs):
        
        super().__init__(
            env=env,
            state_encoder=baseline_state_key,
            **kwargs,
        )
        
class CountingBlackjackAgent(QLearningBlackjackAgent):
    def __init__(
            self, 
            env: gym.Env,
            **kwargs):
        
        super().__init__(
            env=env,
            state_encoder=counting_state_key,
            **kwargs,
        )

In [ ]:
SEEDS = [1, 42, 123]
EPISODES_PER_SEED = 10_000
EVAL_SEEDS = [1234, 4321, 9876]
EVAL_EPISODES = 1_000
CHECKPOINT_INTERVAL = 10_000_000
TRAINING_HISTORY_LIMIT = 100_000
SAVE_FINAL_MODELS = True
MODEL_DIR = PROJECT_ROOT / "models"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

agent_config = {
    "learning_rate": 0.01,
    "initial_epsilon": 1.0,
    "final_epsilon": 0.10,
    "epsilon_decay": (1.0 - 0.10) / (EPISODES_PER_SEED * 0.90),
    "discount_factor": 0.99,
}

def make_env(seed: int, n_episodes: int):
    env = BlackjackEnv(
        num_decks=6,
        penetration=0.75,
        stand_on_soft_17=True,
    )

    env = gym.wrappers.RecordEpisodeStatistics(
        env,
        buffer_length=min(n_episodes, TRAINING_HISTORY_LIMIT),
    )

    env.reset(seed=seed)
    env.action_space.seed(seed)

    return env

agents = {}

for seed in SEEDS:
    agents[f"baseline-{seed}"] = BaselineBlackjackAgent(
        env=make_env(seed, EPISODES_PER_SEED),
        **agent_config,
    )
    agents[f"counting-{seed}"] = CountingBlackjackAgent(
        env=make_env(seed, EPISODES_PER_SEED),
        **agent_config,
    )

def split_agent_name(name: str) -> tuple[str, int]:
    agent_type, seed = name.rsplit("-", 1)
    return agent_type, int(seed)

def group_agents_by_type(agents):
    grouped = defaultdict(list)

    for name, agent in agents.items():
        agent_type, seed = split_agent_name(name)
        grouped[agent_type].append((seed, agent))

    return grouped


In [ ]:
# Checkpoint-Auswahl fuer Training und Evaluation
NO_ARTIFACT = None
IN_MEMORY_ARTIFACT = "__in_memory__"

def artifact_label(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

def artifact_options(agent_name: str, include_fresh: bool = False, include_memory: bool = False):
    options = []
    if include_fresh:
        options.append(("Neu trainieren (kein gespeichertes Modell laden)", NO_ARTIFACT))
    if include_memory:
        options.append(("Aktueller Agent im Speicher", IN_MEMORY_ARTIFACT))

    candidates = []
    final_model_suffix = f"_{agent_name}_agent.pkl"
    candidates.extend(
        path for path in MODEL_DIR.glob("*.pkl")
        if path.name.endswith(final_model_suffix)
    )
    candidates.extend((CHECKPOINT_DIR / agent_name).glob("*.pkl"))
    candidates = sorted(
        {path.resolve(): path for path in candidates}.values(),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    options.extend((artifact_label(path), str(path)) for path in candidates)
    return options

TRAIN_ARTIFACT_SELECTORS = {
    name: widgets.Dropdown(
        options=artifact_options(name, include_fresh=True),
        value=NO_ARTIFACT,
        description=f"{name}:",
        layout=widgets.Layout(width="760px"),
    )
    for name in agents
}

display(widgets.VBox([widgets.HTML("<b>Training fortsetzen ab:</b>"), *TRAIN_ARTIFACT_SELECTORS.values()]))


# Training

In [ ]:
from utils.training import train_single_agent, run_parallel_with_dashboard

if __name__ == "__main__":
    tasks = []
    for name, agent in agents.items():
        agent_type, seed = split_agent_name(name)
        selected_artifact = TRAIN_ARTIFACT_SELECTORS[name].value

        tasks.append((
            name, agent, agent_type, seed, selected_artifact,
            EPISODES_PER_SEED, CHECKPOINT_INTERVAL, CHECKPOINT_DIR / name, RUN_ID, TRAINING_HISTORY_LIMIT
        ))

    results = run_parallel_with_dashboard(
        worker_func=train_single_agent,
        base_tasks=tasks,
        agent_names=list(agents.keys()),
        max_value_per_agent=EPISODES_PER_SEED,
        title="Training"
    )

    for name, trained_agent in results:
        agents[name] = trained_agent

# Saving

In [ ]:
timestamp = RUN_ID

MODEL_DIR.mkdir(parents=True, exist_ok=True)

if SAVE_FINAL_MODELS:
    for name, agent in agents.items():
        agent_type, seed = split_agent_name(name)
        print(f"Saving {name} agent...")
        
        filename = f"{timestamp}_{name}_agent.pkl"
        path = MODEL_DIR / filename
        
        agent.save(
            path,
            label=f"{name}_{timestamp}",
            artifact_type="final",
            episode=EPISODES_PER_SEED,
            n_episodes=EPISODES_PER_SEED,
            base_seed=seed,
            metadata={
                "agent_name": name,
                "agent_type": agent_type,
                "run_id": RUN_ID,
                "seed": seed,
                "target_episode": EPISODES_PER_SEED,
            },
            include_history=False,
        )
        print(f"-> Erfolgreich gespeichert unter: {path}")
else:
    print("Finales Speichern ist deaktiviert (SAVE_FINAL_MODELS=False).")


# Evaluation

## Greedy Policy Evaluation


In [ ]:
# Auswahl, welches Modell/Checkpoint evaluiert werden soll
EVAL_ARTIFACT_SELECTORS = {
    name: widgets.Dropdown(
        options=artifact_options(name, include_memory=True),
        value=IN_MEMORY_ARTIFACT,
        description=f"{name}:",
        layout=widgets.Layout(width="760px"),
    )
    for name in agents
}

display(widgets.VBox([widgets.HTML("<b>Evaluation-Modell auswaehlen:</b>"), *EVAL_ARTIFACT_SELECTORS.values()]))


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from utils.training import run_parallel_with_dashboard
from utils.evaluation import evaluate_single_agent_parallel  # <- Jetzt sauber importiert!
from utils.env_utils import make_blackjack_env             # <- Zentrale Factory nutzen

# =========================================================================
# 1. HILFSFUNKTIONEN & STRATEGIEN
# =========================================================================
AGENT_CLASSES = {
    "baseline": BaselineBlackjackAgent,
    "counting": CountingBlackjackAgent,
}

def make_agent(agent_name: str, seed: int):
    agent_type, _ = split_agent_name(agent_name)
    return AGENT_CLASSES[agent_type](
        env=make_blackjack_env(seed=seed, n_episodes=EVAL_EPISODES), # <- Korrigiert
        **agent_config,
    )

def selected_eval_agent(agent_name: str):
    selected_artifact = EVAL_ARTIFACT_SELECTORS[agent_name].value
    if selected_artifact == IN_MEMORY_ARTIFACT:
        return agents[agent_name], "in_memory"

    _, seed = split_agent_name(agent_name)
    eval_agent = make_agent(agent_name, seed=seed)
    artifact = eval_agent.load(selected_artifact)
    label = artifact.get("label") or artifact_label(Path(selected_artifact))
    return eval_agent, label


def eval_task_name(agent_name: str, eval_seed: int) -> str:
    return f"{agent_name}__eval_seed_{eval_seed}"


def split_eval_task_name(task_name: str) -> tuple[str, int]:
    agent_name, eval_seed = task_name.rsplit("__eval_seed_", 1)
    return agent_name, int(eval_seed)


# =========================================================================
# 2. HAUPTPROZESS (Aufgaben vorbereiten und Runner starten)
# =========================================================================
if __name__ == "__main__":
    tasks = []
    agent_names = []
    
    # Modelle laden / vorbereiten
    for name in agents:
        eval_agent, source_label = selected_eval_agent(name)
        agent_type, _ = split_agent_name(name)
        
        # Bestimme die richtige Encoder-Funktion direkt hier im Hauptthread
        encoder_func = baseline_state_key if agent_type == "baseline" else counting_state_key
        
        for eval_seed in EVAL_SEEDS:
            task_name = eval_task_name(name, eval_seed)
            agent_names.append(task_name)
            tasks.append((
                task_name,
                eval_agent.q_values,
                encoder_func,
                source_label,
                EVAL_EPISODES,
                eval_seed,
            ))
        
    # Den generischen Dashboard-Runner aufrufen
    results = run_parallel_with_dashboard(
        worker_func=evaluate_single_agent_parallel,
        base_tasks=tasks,
        agent_names=agent_names,
        max_value_per_agent=EVAL_EPISODES,
        title="Evaluation"
    )
    
    # Ergebnisse aus dem Pool wieder einsammeln
    greedy_eval_results = {}
    for task_name, metrics in results:
        agent_name, eval_seed = split_eval_task_name(task_name)
        agent_type, train_seed = split_agent_name(agent_name)
        metrics.update({
            "agent_name": agent_name,
            "agent_type": agent_type,
            "train_seed": train_seed,
            "eval_seed": eval_seed,
        })
        greedy_eval_results[task_name] = metrics

    # DataFrame bauen, speichern und anzeigen
    greedy_eval_df = pd.DataFrame(greedy_eval_results).T
    EVALUATION_DIR = MODEL_DIR / "evaluations" / RUN_ID
    EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
    greedy_eval_df.to_csv(EVALUATION_DIR / "greedy_metrics.csv", index_label="eval_run")
    greedy_eval_df.to_json(EVALUATION_DIR / "greedy_metrics.json", orient="index", indent=2)
    print(f"Evaluation gespeichert unter: {EVALUATION_DIR}")
    display(greedy_eval_df)


## Training Dynamics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def get_moving_stats(arr, window: int):
    arr = np.asarray(arr).flatten()
    if len(arr) == 0:
        return arr, arr, np.arange(0)

    min_periods = max(1, window // 10)
    series = pd.Series(arr)

    means = series.rolling(window=window, min_periods=min_periods).mean().to_numpy()
    stds = series.rolling(window=window, min_periods=min_periods).std().to_numpy()
    stds = np.nan_to_num(stds)

    return means, stds, np.arange(len(arr))


def plot_training_rewards(agents: dict, window: int = 1000):
    plt.figure(figsize=(8, 6)) # Explicitly create a new figure
    grouped = group_agents_by_type(agents)
    
    for agent_type, agent_list in grouped.items():
        style = AGENT_STYLES[agent_type]
        reward_curves = []
        for seed, agent in agent_list:
            rewards = np.asarray(agent.episode_rewards)
            means, _, _ = get_moving_stats(rewards, window)
            reward_curves.append(means)
            
        # align lengths
        min_len = min(len(c) for c in reward_curves)
        reward_curves = np.array([c[:min_len] for c in reward_curves])
        reward_mean = reward_curves.mean(axis=0)
        reward_std = reward_curves.std(axis=0)
        x_r = np.arange(len(reward_mean))

        plt.plot(
            x_r,
            reward_mean,
            label=style["label"],
            color=style["color"],
            linewidth=2.5,
        )

        plt.fill_between(
            x_r,
            reward_mean - reward_std,
            reward_mean + reward_std,
            color=style["color"],
            alpha=0.2,
        )
        
    plt.title("Episode Rewards (Mittelwert über Seeds)")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.legend()

    save_fig("training_results_combined_rewards")
    plt.show()
    plt.close() 
    

def plot_training_td_error(agents: dict, window: int = 5000):
    plt.figure(figsize=(8, 6)) # Explicitly create a new figure
    grouped = group_agents_by_type(agents)

    for agent_type, agent_list in grouped.items():
        style = AGENT_STYLES[agent_type]
        td_curves = []
        for seed, agent in agent_list:
            td_errors = np.asarray(agent.training_error)
            means, _, _ = get_moving_stats(td_errors, window)
            td_curves.append(means)
            
        # align lengths
        min_len = min(len(c) for c in td_curves)
        td_curves = np.array([c[:min_len] for c in td_curves])
        td_mean = td_curves.mean(axis=0)
        td_std = td_curves.std(axis=0)
        x_t = np.arange(len(td_mean))

        plt.plot(
            x_t,
            td_mean,
            label=style["label"],
            color=style["color"],
            linewidth=2.5,
        )

        plt.fill_between(
            x_t,
            td_mean - td_std,
            td_mean + td_std,
            color=style["color"],
            alpha=0.2,
        )

    plt.title("TD Error (Mittelwert über Seeds)")
    plt.xlabel("Episode")
    plt.ylabel("TD Error")
    plt.legend()

    save_fig("training_results_combined_td")
    plt.show()
    plt.close()

In [ ]:
plot_training_rewards(agents, window=1000)
plot_training_td_error(agents, window=5000)

## Performance Comparison

In [ ]:
def get_reward_matrix(agent_list):
    return np.vstack([
        np.asarray(agent.episode_rewards)
        for seed, agent in agent_list
    ])


def flatten_rewards(agent_list):
    return get_reward_matrix(agent_list).ravel()


def calculate_metrics(rewards: np.ndarray) -> dict:
    rewards = np.asarray(rewards)

    wins = np.sum(rewards == 1)
    losses = np.sum(rewards == -1)
    pushes = np.sum(rewards == 0)
    total = len(rewards)

    return {
        "win_rate": wins / total if total > 0 else 0.0,
        "loss_rate": losses / total if total > 0 else 0.0,
        "push_rate": pushes / total if total > 0 else 0.0,
        "average_reward": np.mean(rewards) if total > 0 else 0.0,
        "std_reward": np.std(rewards) if total > 0 else 0.0,
    }


grouped = group_agents_by_type(agents)

per_seed_rows = []
for agent_type, agent_list in grouped.items():
    for seed, agent in agent_list:
        row = calculate_metrics(np.asarray(agent.episode_rewards))
        row.update({"agent_type": agent_type, "seed": seed})
        per_seed_rows.append(row)

per_seed_metrics_df = pd.DataFrame(per_seed_rows).sort_values(["agent_type", "seed"])
display(per_seed_metrics_df)

baseline_rewards = flatten_rewards(grouped["baseline"])
counting_rewards = flatten_rewards(grouped["counting"])

baseline_metrics = calculate_metrics(baseline_rewards)
counting_metrics = calculate_metrics(counting_rewards)

summary_metrics_df = pd.DataFrame({
    "baseline": baseline_metrics,
    "counting": counting_metrics,
}).T
display(summary_metrics_df)

for name, metrics in [("Baseline", baseline_metrics), ("Counting", counting_metrics)]:
    print(f"{name} metrics:")
    print(f"  Win Rate:       {metrics['win_rate']:.1%}")
    print(f"  Loss Rate:      {metrics['loss_rate']:.1%}")
    print(f"  Push Rate:      {metrics['push_rate']:.1%}")
    print(f"  Avg Reward:     {metrics['average_reward']:.3f}")
    print(f"  Std Reward:     {metrics['std_reward']:.3f}")
    print()


In [ ]:
from typing import cast

greedy_eval_data = cast(pd.DataFrame | None, globals().get("greedy_eval_df"))
if greedy_eval_data is None:
    raise RuntimeError("Bitte zuerst die Greedy-Evaluation ausführen.")

evaluation_dir = globals().get("EVALUATION_DIR")


print("Zusammenfassung pro Seed:")
display(greedy_eval_data[["agent_name", "train_seed", "eval_seed", "source", "episodes", "win_rate", "loss_rate", "push_rate", "average_reward"]])

from utils.evaluation import calculate_evaluation_cis

print("\nBerechne mathematische 95% Konfidenzintervalle:")
greedy_eval_ci_df = calculate_evaluation_cis(greedy_eval_data)
display(greedy_eval_ci_df)
if evaluation_dir is not None:
    greedy_eval_ci_df.to_csv(evaluation_dir / "greedy_metrics_ci.csv", index_label="eval_run")

In [ ]:
print("\nGeneriere Signifikanz-Plots...")
greedy_eval_plot_df = greedy_eval_data.drop(
    columns=["agent_name", "agent_type", "train_seed", "eval_seed"],
    errors="ignore",
)

from utils.evaluation import plot_evaluation_metrics_with_cis

plot_evaluation_metrics_with_cis(
    greedy_eval_df=greedy_eval_plot_df,
    agent_styles=AGENT_STYLES,
    split_agent_name_func=lambda task_name: split_agent_name(split_eval_task_name(task_name)[0]),
    save_fig_func=save_fig
)

In [ ]:
from utils.evaluation import (
    TRUE_COUNT_BUCKET_SUMMARY_COLUMNS,
    build_true_count_bucket_tasks,
    create_true_count_bucket_widget,
    evaluate_true_count_buckets_parallel,
    prepare_true_count_bucket_reports,
)

TRUE_COUNT_BUCKETS = [
    ("<= -3", None, -3),
    ("-2 to 0", -2, 0),
    ("1 to 2", 1, 2),
    (">= 3", 3, None),
]

if "EVAL_ARTIFACT_SELECTORS" not in globals():
    raise RuntimeError("Bitte zuerst die Evaluation-Artefakte auswählen.")

bucket_tasks, bucket_agent_names = build_true_count_bucket_tasks(
    agents=agents,
    selected_eval_agent_func=selected_eval_agent,
    split_agent_name_func=split_agent_name,
    eval_task_name_func=eval_task_name,
    baseline_state_key_func=baseline_state_key,
    counting_state_key_func=counting_state_key,
    eval_seeds=EVAL_SEEDS,
    eval_episodes=EVAL_EPISODES,
    buckets=TRUE_COUNT_BUCKETS,
)

bucket_results = run_parallel_with_dashboard(
    worker_func=evaluate_true_count_buckets_parallel,
    base_tasks=bucket_tasks,
    agent_names=bucket_agent_names,
    max_value_per_agent=EVAL_EPISODES,
    title="True-Count-Bucket-Evaluation",
)

evaluation_dir = globals().get("EVALUATION_DIR")
if evaluation_dir is None:
    evaluation_dir = MODEL_DIR / "evaluations" / RUN_ID
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    EVALUATION_DIR = evaluation_dir

true_count_bucket_raw_df, true_count_bucket_summary_df, true_count_bucket_seed_df = prepare_true_count_bucket_reports(
    bucket_results=bucket_results,
    split_eval_task_name_func=split_eval_task_name,
    split_agent_name_func=split_agent_name,
    evaluation_dir=evaluation_dir,
)

print(f"True-Count-Bucket-Auswertung gespeichert unter: {evaluation_dir}")
display(true_count_bucket_summary_df[TRUE_COUNT_BUCKET_SUMMARY_COLUMNS])
display(create_true_count_bucket_widget(
    summary_df=true_count_bucket_summary_df,
    seed_df=true_count_bucket_seed_df,
    buckets=TRUE_COUNT_BUCKETS,
))


# Policy & Q-Value Visualization
Hier analysieren wir interaktiv die gelernten Strategien der Agenten und vergleichen sie mit der Basic Strategy.

In [ ]:
import ipywidgets as widgets
from utils.visualizations import plot_policy_and_q_values

def interactive_plot_wrapper(agent_name, true_count):
    if agent_name in agents:
        plot_policy_and_q_values(
            agent=agents[agent_name], 
            agent_name=agent_name, 
            split_agent_name_func=split_agent_name, 
            save_fig_func=save_fig, 
            true_count=true_count
        )

# --- STEUERUNG FÜR AGENT A (LINKS) ---
agent_selector_a = widgets.Dropdown(options=list(agents.keys()), value=list(agents.keys())[0], description='Agent A:')
true_count_slider_a = widgets.IntSlider(value=0, min=-10, max=10, step=1, description='True Count A:', disabled=True)

def update_slider_visibility_a(*args):
    agent_type, _ = split_agent_name(agent_selector_a.value)
    true_count_slider_a.disabled = (agent_type == "baseline")

agent_selector_a.observe(update_slider_visibility_a, 'value')
plot_a = widgets.interactive(interactive_plot_wrapper, agent_name=agent_selector_a, true_count=true_count_slider_a)


# --- STEUERUNG FÜR AGENT B (RECHTS) ---
# Startwert auf den letzten Eintrag gesetzt, damit direkt zwei unterschiedliche Agenten laden
agent_selector_b = widgets.Dropdown(options=list(agents.keys()), value=list(agents.keys())[-1], description='Agent B:')
true_count_slider_b = widgets.IntSlider(value=0, min=-10, max=10, step=1, description='True Count B:', disabled=True)

def update_slider_visibility_b(*args):
    agent_type, _ = split_agent_name(agent_selector_b.value)
    true_count_slider_b.disabled = (agent_type == "baseline")

agent_selector_b.observe(update_slider_visibility_b, 'value')
plot_b = widgets.interactive(interactive_plot_wrapper, agent_name=agent_selector_b, true_count=true_count_slider_b)

update_slider_visibility_a()
update_slider_visibility_b()

widgets.HBox([plot_a, plot_b])